# Week 6 Exercise — Price-from-Description Predictor

**"The Price is Right"** — Predict a product's price in USD from its text description using a frontier LLM (zero-shot). Includes a small in-notebook eval set and a **Gradio** UI.

Set `OPENROUTER_API_KEY` in `.env`. Optional: run Ollama locally for the Ollama model.

In [ ]:
import os
import re
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [ ]:
load_dotenv(override=True)
api_key = os.getenv("OPENROUTER_API_KEY")

if not api_key:
    print("Add OPENROUTER_API_KEY to .env")
else:
    print("OpenRouter API key loaded.")

MODELS = {
    "OpenRouter: gpt-4o-mini": ("openai/gpt-4o-mini", "https://openrouter.ai/api/v1", api_key),
    "OpenRouter: claude-3-haiku": ("anthropic/claude-3-haiku", "https://openrouter.ai/api/v1", api_key),
    "Ollama: llama3.2": ("llama3.2", "http://localhost:11434/v1", "ollama"),
}

def get_client(model_key):
    model_id, base_url, key = MODELS.get(model_key, list(MODELS.values())[0])
    return OpenAI(api_key=key or "ollama", base_url=base_url), model_id

## Eval set

Small list of (description, actual_price) for computing MAE. No dependency on week6/pricer or HuggingFace.

In [ ]:
EVAL_EXAMPLES = [
    ("Wireless Bluetooth earbuds, 24hr battery, noise cancelling", 49.99),
    ("Stainless steel water bottle 32oz insulated", 29.99),
    ("Mechanical keyboard RGB backlit, cherry MX switches", 129.00),
    ("USB-C laptop docking station dual 4K HDMI", 189.99),
    ("Portable power bank 20000mAh PD fast charge", 45.00),
    ("Standing desk mat anti-fatigue", 39.95),
    ("4K webcam with ring light and microphone", 79.99),
    ("Ergonomic office chair mesh back adjustable", 249.00),
    ("Smart thermostat WiFi programmable", 89.00),
    ("Noise cancelling over-ear headphones Bluetooth 5.0", 199.00),
]

print(f"Eval set: {len(EVAL_EXAMPLES)} (description, price) pairs")

## Predictor

Prompt the LLM to estimate price in USD; respond with the number only. Parse the reply to extract a float.

In [ ]:
PROMPT_TEMPLATE = """Estimate the price of this product in USD. Respond with only the price number, no explanation.

{description}"""

def parse_price(text):
    if not text:
        return 0.0
    text = text.strip().replace(",", "")
    match = re.search(r"(\\d+(?:\\\\.\\d+)?)", text)
    if match:
        try:
            return max(0.0, float(match.group(1)))
        except ValueError:
            pass
    return 0.0

def predict_price(description, model_key):
    if not description or not description.strip():
        return 0.0
    client, model_id = get_client(model_key)
    prompt = PROMPT_TEMPLATE.format(description=description.strip())
    try:
        r = client.chat.completions.create(
            model=model_id,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=15,
        )
        content = (r.choices[0].message.content or "").strip()
        return round(parse_price(content), 2)
    except Exception as e:
        return f"Error: {e}"

In [ ]:
def evaluate_mae(model_key):
    errors = []
    for desc, actual in EVAL_EXAMPLES:
        pred = predict_price(desc, model_key)
        if isinstance(pred, (int, float)):
            errors.append(abs(pred - actual))
    if not errors:
        return "Could not compute (all predictions failed or errored)."
    mae = sum(errors) / len(errors)
    return f"MAE over {len(errors)} examples: ${mae:.2f}"

print(evaluate_mae(list(MODELS.keys())[0]))

## Gradio UI

Enter a product description and get a predicted price. Optionally run evaluation to see MAE on the in-notebook set.

In [ ]:
def run_eval(model_key):
    return evaluate_mae(model_key)

model_dd = gr.Dropdown(choices=list(MODELS.keys()), value=list(MODELS.keys())[0], label="Model")

with gr.Blocks(title="Week 6 — Price from Description") as demo:
    gr.Markdown("## The Price is Right — Predict product price from description")
    with gr.Row():
        inp = gr.Textbox(label="Product description", lines=3, placeholder="e.g. Wireless Bluetooth earbuds, 24hr battery...")
        out = gr.Number(label="Predicted price ($)")
    with gr.Row():
        model_dd.render()
        btn = gr.Button("Predict")
    btn.click(fn=lambda d, m: predict_price(d, m), inputs=[inp, model_dd], outputs=out)
    gr.Markdown("### Evaluation on in-notebook set")
    eval_btn = gr.Button("Run MAE evaluation")
    eval_out = gr.Textbox(label="MAE")
    eval_btn.click(fn=run_eval, inputs=[model_dd], outputs=eval_out)

demo.launch()